# 🤖 Agentic AI with Bigdata.com Search API

This notebook demonstrates how your AI agents can interact with **Bigdata.com Search API** to access real-time market intelligence, news, filings, and earnings transcripts—combined with your internal data sources.

## What This Demonstrates

This notebook delivers a cited, multi-source answer in one flow—combining your internal portfolios and research with **Bigdata.com** for the external side: **Knowledge Graph** resolves companies to entity IDs for precise filtering, and the **Search API** delivers news, filings, and transcripts with sentiment. Your agent leans on Bigdata.com for real-time market intelligence and entity-scoped search, then weaves in internal data for a single, traceable response with inline source links.

**Bigdata.com Integration Patterns:**
- **Knowledge Graph API** → Resolve company names to entity IDs for precise filtering
- **Search API** → Query news, SEC filings, earnings transcripts with sentiment scores
- **Tool-based Architecture** → Wrap APIs as callable tools for any agentic framework

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine external market data with proprietary insights

**Framework Flexibility:**
> This demo uses **LangChain** and **LangSmith** for observability, but the integration pattern applies to any agentic framework: **CrewAI**, **AutoGen**, **Google A2A**, or custom implementations. The key is wrapping Bigdata.com APIs as tools your agents can call.

---

## Architecture

![Agent to Bigdata APIs ](./static/agent-search.png)

**Key Points:**
- **Single agent interface** for internal DB, vector store, and Bigdata.com Search/Knowledge Graph
- **Production-ready** tooling: retry, logging, KG entity cache
- **Observability** via LangSmith for tool selection and latency
- **Flexibility**: Same pattern works with CrewAI, AutoGen, or custom frameworks—wrap APIs as tools

---

## 1️⃣ Install Dependencies

Install from the project root before running this notebook:

```bash
uv sync
```

In [ ]:
# Dependencies: install from project root with uv sync (see README)

## 2️⃣ Import Libraries

Import core utilities and display helpers from `langgraph_core`.

In [2]:
import os
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML

# LangChain
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI

# Local utilities - environment, data sources, and display helpers
import sys
sys.path.append('.')
from langgraph_core import (
    # Environment & Data Setup
    setup_environment,
    create_financial_database,
    create_vector_store,
    
    # Tool Providers
    get_bigdata_tools,
    get_database_tools,
    get_vectorstore_tools,
    
    # Display Utilities
    display_query,
    display_response,
    display_tools_used,
    display_citations
)

# Load environment variables
load_dotenv()

print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [3]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project='bigdata-agent-demo',
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")
print("\n🔗 View traces at: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)

🔗 View traces at: https://smith.langchain.com


## 4️⃣ Load Local Tools

Load tools that interact with local data sources.

In [4]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for tool in local_db_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for tool in local_vector_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transaction...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity...


## 5️⃣ Load Bigdata.com Tools

Load external tools for market intelligence:

- **Knowledge Graph API** - Entity lookup and company identification
- **Search API** - News, filings, transcripts with sentiment
- **Other** - can be added based on need

In [5]:
# Get Bigdata.com tools
bigdata_tools = get_bigdata_tools()
print(f"✅ Loaded {len(bigdata_tools)} Bigdata.com tools:")
for tool in bigdata_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 Bigdata.com tools:
   - bigdata_lookup_company: Look up a company's Bigdata entity ID using the Knowledge Gr...
   - bigdata_search_news: Search Bigdata.com for financial news and market intelligenc...


## 6️⃣ Combine All Tools

Merge tools from all sources for the agent.

In [6]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata.com tools: {len(bigdata_tools)}")


✅ Total tools available: 5
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata.com tools: 2


## 7️⃣ Create LangChain Agent

Create an agent with all tools configured.

**ReAct Pattern:**
- **Reasoning**: Plans which tools to use based on query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

**Note:** Update it based on need.

In [7]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com APIs):**
1. `bigdata_lookup_company` - Look up company entity IDs from the Knowledge Graph
2. `bigdata_search_news` - Search financial news and market intelligence with source citations

**Internal Data (Company Systems):**
3. `internal_query_database` - Execute SQL queries on portfolio/transaction database
4. `internal_portfolio_summary` - Get portfolio holdings and performance summary
5. `internal_search_research` - Search internal investment research documents

Guidelines:
- For company news, first use `bigdata_lookup_company` to get the entity_id, then use it in `bigdata_search_news`
- For portfolio questions, use `internal_portfolio_summary` or `internal_query_database` with SQL
- Combine external market data with internal holdings/research for comprehensive analysis

**Citation format:** Use inline citations with the **source name as the link text** (not the raw URL). Format as markdown: [Source Name](url) or [1](url), [2](url) so the reader sees a clickable source name. Do not paste full URLs in the body.

**Do not add a separate "Sources" or "References" block at the end** when you have already used inline citations. Inline citations are sufficient. For internal data: Briefly mention "From internal database" or "According to internal research."
**Do not offer suggestions for follow up questions**

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

# Initialize LLM
model = "gpt-5" 
llm = ChatOpenAI(
    model=model,
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create agent using langchain.agents.create_agent
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: {model}")
print(f"   System prompt configured")
print("\n🔍 Agent Tool Selection:")
print("   • Portfolio/holdings → internal_query_database")
print("   • Internal research → internal_search_research")
print("   • Company lookup → bigdata_lookup_company")
print("   • Market news → bigdata_search_news")

✅ Agent created with 5 tools
   Model: gpt-5
   System prompt configured

🔍 Agent Tool Selection:
   • Portfolio/holdings → internal_query_database
   • Internal research → internal_search_research
   • Company lookup → bigdata_lookup_company
   • Market news → bigdata_search_news


## 8️⃣ System Prompt

The agent uses the `SYSTEM_PROMPT` defined above. Key elements:

**External Tools (Bigdata.com):**
- `bigdata_lookup_company` - Get entity IDs for filtering news
- `bigdata_search_news` - Search financial news with citations

**Internal Tools:**
- `internal_query_database` - SQL queries on portfolios
- `internal_portfolio_summary` - Holdings & performance
- `internal_search_research` - Semantic search on research docs

**Guidelines:** Look up entity_id first, then search news; use internal tools for portfolio questions; combine sources for comprehensive analysis.

**Citation:** Inline citations with **source name as hyperlink** — format as [Source Name](url). Do not add a separate "Sources" block; citations are rendered inline.

---

## 9️⃣ Example Queries

Run queries that combine data from multiple sources.

### Query 1: Multi-Source NVIDIA Analysis

Combines internal holdings, research, and external news.

In [8]:
query = """I need a comprehensive analysis of NVIDIA:
1. What's our current position in NVIDIA across all portfolios?
2. What does our internal research say about NVIDIA's investment thesis?
3. What's the latest news about NVIDIA from market sources?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a consolidated view across our systems and external market sources.

1) Our current NVIDIA position (from internal database)
- Aggregate across portfolios:
  - Shares: 20,000
  - Last marked price: $875.50
  - Market value: $17,510,000
  - Aggregate cost basis: ~$9,560,000 (weighted avg. cost ~$478/share)
  - Unrealized P&L: ~$7,950,000
  - Share of total firm AUM (PF001–PF003: $95M): ~18.4%
- By portfolio:
  - PF001 (US Large Cap Growth): No NVDA position
  - PF002 (AI & Semiconductor Focus):
    - Shares: 12,000
    - Market value: $10,506,000
    - Avg. cost: $450
    - Unrealized P&L: $5,106,000
    - Portfolio weight: ~70.0% (AUM $15M)
  - PF003 (Diversified Tech Leaders):
    - Shares: 8,000
    - Market value: $7,004,000
    - Avg. cost: $520
    - Unrealized P&L: $2,844,000
    - Portfolio weight: ~14.0% (AUM $50M)

2) Internal research summary on NVIDIA’s investment thesis
According to internal research:
- Core thesis and drivers (Dec 15, 2024 thesis update):
  - Data center AI leadership: Demand for H100/H200 has driven outsized growth; next-gen Blackwell (B100/B200) expected to deliver ~2.5x performance and extend leadership.
  - Software moat: CUDA ecosystem (4M+ developers) creates high switching costs versus alternatives, underpinning platform durability.
  - Expanding TAM: AI inference opportunity modeled at ~$150B by 2027 as enterprises scale deployment; networking (Mellanox/InfiniBand) and full-stack systems (e.g., DGX) reinforce end-to-end positioning.
  - Valuation view at the time: Price target $950 (25x FY26E EPS); rating: Strong Buy.
- Strategy guidance (Jan 5, 2025):
  - Recommended +3% overweight in tech allocation due to continued supply/demand tightness in AI training capacity and strong order visibility.
- Key risks (Jan 10, 2025 risk assessment):
  - China export controls: 20–25% of revenue at risk if restrictions tighten or mitigation underperforms.
  - Competition: AMD (MI300/MI400) and custom silicon (e.g., TPUs) could pressure share and pricing over time; ROCm ecosystem improving but still trails CUDA.
  - Supply chain: HBM and advanced packaging constraints remain gating factors; any disruption could affect deliveries.
  - Valuation/AI cycle risk: Elevated multiples across mega-cap tech and a potential mismatch between AI infrastructure buildout and near-term monetization.
(From internal research)

3) Latest market news on NVIDIA (last 30 days)
- Valuation debate and long-term AI spend backdrop:
  - Discussion around NVIDIA trading at a premium multiple, with some estimates citing multi-trillion annual AI data center capex potential by 2030, reinforcing long-term demand assumptions [Nasdaq](https://www.nasdaq.com/articles/nvidia-41x-forward-earnings-buy-hold-or-cash-out).
- Upcoming earnings as a catalyst:
  - February 25 earnings flagged as pivotal for direction; bull case hinges on visible Blackwell ramp, >70% gross margins, and sustained China orders; bears focused on valuation risk if guide disappoints [AOL.com](https://www.aol.com/finance/200-150-nvidia-february-25-110041647.html).
- Ecosystem and capacity buildout:
  - NVIDIA reportedly investing $2B in CoreWeave to support AI data center buildouts (targeting 5GW by 2030); reports also note CEO Jensen Huang’s China visit—potentially relevant for China demand and policy read-throughs [MSN](https://www.msn.com/en-us/money/markets/nvidia-stock-gains-what-it-needs-from-big-tech-earnings/ar-AA1UZYbe?ocid=finance-verthp-feeds).
- China export landscape:
  - Headlines suggest China has opened the door to certain H200 shipments, though with lingering uncertainties; notes also reference AMD’s MI400 shipping alongside NVIDIA systems as competitive context [AOL.com](https://www.aol.com/finance/china-opens-door-nvidia-h200-143934877.html).
- Product cadence and demand signals:
  - Commentary highlights strong Blackwell demand; management tone cited as constructive with indications cloud GPUs remain effectively sold out into the next product cycle; earnings timing reiterated as late February [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-eyes-openai-investment-buy-182748007.html).

Implications for our position
- Near-term: Earnings/guidance later in February is the key stock catalyst for our sizable PF002/PF003 exposure. Watch for clarity on Blackwell ramp timing, margin profile, and China demand.
- Medium term: Investment in partner capacity (e.g., CoreWeave) and persistent tightness in AI compute should support orders, but competitive dynamics (AMD/custom silicon) and any renewed export constraints remain the primary risks.
- Positioning context: PF002’s concentration (~70%) magnifies outcome sensitivity to the upcoming print and guide.

### Query 2: Portfolio Risk Assessment

Analyzes portfolio holdings with internal research and external news.

In [9]:
query = """Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. Are there any recent news events affecting these holdings?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise status check on PF002.

1) Current holdings and performance (From internal database)
- NVIDIA (NVDA)
  - Shares: 12,000 | Avg cost: 450.00 | Price: 875.50
  - Market value: $10,506,000 | Unrealized P&L: +$5,106,000 (+94.6%)
  - Portfolio weight: ~65.7%
- Broadcom (AVGO)
  - Shares: 1,500 | Avg cost: 850.00 | Price: 1,425.00
  - Market value: $2,137,500 | Unrealized P&L: +$862,500 (+67.6%)
  - Portfolio weight: ~13.4%
- Palantir (PLTR)
  - Shares: 25,000 | Avg cost: 18.50 | Price: 65.25
  - Market value: $1,631,250 | Unrealized P&L: +$1,168,750 (+252.6%)
  - Portfolio weight: ~10.2%
- AMD (AMD)
  - Shares: 8,000 | Avg cost: 95.00 | Price: 145.25
  - Market value: $1,162,000 | Unrealized P&L: +$402,000 (+52.9%)
  - Portfolio weight: ~7.3%
- Taiwan Semi (TSM)
  - Shares: 3,000 | Avg cost: 110.00 | Price: 185.75
  - Market value: $557,250 | Unrealized P&L: +$227,250 (+68.9%)
  - Portfolio weight: ~3.5%
- Portfolio totals: Market value ≈ $15.994M; Unrealized P&L ≈ +$7.767M; return on cost ≈ +94.4%; 5 holdings. Concentration: NVDA ~66% of portfolio.

2) Key risks flagged by our internal research (According to internal research)
- Valuation/compression risk: AI leaders (esp. NVDA, PLTR) trade at rich multiples; any slowdown in AI monetization or capex could compress multiples.
- AI accelerator competition: AMD MI300X and custom silicon from hyperscalers may pressure NVDA pricing/mix; software ecosystem (CUDA vs. ROCm) is a moat but a focal point of competitive risk.
- Supply constraints: Advanced packaging and HBM (e.g., CoWoS capacity) remain tight; any disruption could cap upside for NVDA/AMD and ripple to TSM.
- China/export controls: U.S. restrictions create demand uncertainty for NVDA; medium risk exposure to China demand and licensing timing.
- Geopolitics/Taiwan risk: Concentration of advanced foundry capacity at TSM exposes the portfolio to geopolitical shocks.
- Datacenter capex sensitivity: Broad slowdowns or ROI scrutiny on AI spending would hit NVDA/AMD/AVGO, and indirectly PLTR (if enterprise deployments defer).
- Customer concentration/inventory cycles: Hyperscaler order concentration and potential inventory corrections can amplify volatility.
- Product and execution risk: Next‑gen GPU roadmaps (e.g., NVDA Blackwell) and software adoption timelines (ROCm, PLTR AIP commercialization) carry timing/delivery risk.
- Regulatory/antitrust backdrop: Sector-wide scrutiny could alter behaviors or cost structures over time; PLTR’s government exposure adds contract timing/budget risks.

3) Recent news affecting these holdings (last 30 days)
- NVIDIA
  - China approved initial imports of Nvidia’s H200 AI chips, seen as a modest tailwind amid export-control constraints [Yahoo! Finance](https://uk.finance.yahoo.com/news/nvidia-stock-rises-china-approves-144922818.html).
  - Street commentary continues to emphasize Nvidia’s dominant AI accelerator share and sticky CUDA ecosystem; ongoing debate around China exposure persists [Yahoo! Finance](https://finance.yahoo.com/news/jpmorgan-revamps-ai-stocks-buy-210700435.html).
  - Sector-wide profit-taking earlier in the month hit AI chip names and highlighted persistent China/geopolitical overhangs [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-amd-intel-plunges-amid-165133726.html).
- Palantir
  - Pre‑earnings focus on whether U.S. commercial growth is accelerating and margins remain durable as AIP scales [Business Insider](https://markets.businessinsider.com/news/stocks/pltr-vs-smci-which-ai-stock-is-the-better-buy-ahead-of-earnings-top-analysts-weigh-in-1035743560).
  - Ongoing coverage of defense and large multi‑year contracts supports narrative of growing AI adoption in mission‑critical settings [Nasdaq](https://www.nasdaq.com/articles/zacks-analyst-blog-highlights-nvidia-lockheed-martin-ibm-microsoft-and-palantir).
- Competitive/market context (indirect but relevant to NVDA/AMD/TSM/AVGO)
  - Reports that Alibaba may pursue an IPO of its AI chip unit underscore intensifying AI hardware competition in China, a factor for NVDA and upstream suppliers [Yahoo! Finance](https://finance.yahoo.com/news/alibaba-shares-surge-report-co-094300227.html).
- AVGO, AMD, TSM
  - No material company‑specific headlines surfaced in the last 30 days via our news search filters. These positions remain primarily driven by sector AI‑capex trends, supply chain conditions (HBM/advanced packaging), and upcoming earnings/guidance.

If you want, I can refresh the news feed again closer to earnings windows for each name and flag any updates.

### Query 3: News Sentiment Analysis

Focuses on external market intelligence with sentiment.

In [10]:
query = """What's the market sentiment around Apple in the last 30 days?
Look for news about:
1. Product launches and innovations
2. Financial performance
3. Regulatory issues

Also check if we hold Apple in any portfolios."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise sentiment read on Apple over the last 30 days, organized by your topics, plus our internal exposure.

Overall sentiment (30-day): Moderately positive. Strong earnings and iPhone momentum outweigh a persistent EU regulatory overhang; innovation headlines are a net positive but mixed due to lingering AI execution questions.

1) Product launches and innovations
- AI and M&A: Apple reportedly acquired AI startup Q.ai, fueling expectations for on-device AI and wearables integrations (AirPods, Watch, Vision Pro) [CNBC](https://www.cnbc.com/2026/01/29/apple-acquires-israeli-startup-qai-.html), [9to5Mac](https://9to5mac.com/2026/01/29/apple-acquires-secretive-q%e2%80%a4ai-startup-for-2-billion/).
- New hardware/accessories: Apple introduced a new AirTag with expanded connectivity and improved findability [WebWire](https://www.webwire.com/ViewPressRel.asp?aId=349701).
- Creative software push: Apple announced “Apple Creator Studio,” positioning its platforms more squarely around pro/creator workflows [Associated Press](https://apnews.com/press-release/business-wire/apple-introduces-apple-creator-studio-an-inspiring-collection-of-the-most-powerful-creative-apps-0d2b608254304095a542419d0c8bfc28).
- AI product roadmap sentiment: Investor focus remains on Apple’s AI integration and Siri reboot; some skepticism persists about pace relative to peers and Vision Pro’s niche status [Business Insider](https://www.businessinsider.com/apple-earnings-live-updates-ai-gemini-iphone-sales-2026-1?r=UK&IR=T), [AOL.com](https://www.aol.co.uk/articles/investors-look-apple-ai-plans-195904620.html).

Net: Slightly positive—tangible updates (AirTag, creator tools, AI M&A) and mounting expectations on Apple Intelligence, offset by questions about timing and differentiation.

2) Financial performance
- Beat and record iPhone quarter: Apple’s quarter beat expectations with revenue about $143.8B and EPS ahead of estimates; iPhone drove a record quarter [MSN](https://www.msn.com/en-gb/money/technology/apple-sales-profit-beat-wall-street-estimates-amid-staggering-iphone-demand/ar-AA1Vhb61?ocid=finance-verthp-feeds), [Tech Times](https://www.techtimes.com/articles/314378/20260129/apple-generates-over-85-billion-revenue-its-best-ever-quarter-iphones.htm).
- Scale and margins: Installed base surpassed 2.5B devices; product gross margins expanded (cited ~40.7%)—supportive for sentiment and Services monetization [CNBC](https://www.cnbc.com/2026/01/29/whats-next-for-apple-stock-after-big-iphone-sales-everywhere-even-china.html).
- Services: Continued growth and high-margin mix remain a structural positive, though the growth rate may be normalizing off strong comps [CNBC](https://www.cnbc.com/2026/01/29/whats-next-for-apple-stock-after-big-iphone-sales-everywhere-even-china.html).
- Stock reaction and outlook: Shares rose post-call; management commentary pointed to ongoing growth into the March quarter [Morningstar](https://www.morningstar.com/news/marketwatch/20260129460/apples-stock-rises-as-tim-cook-gives-just-enough-detail-on-wall-streets-most-burning-question), [AOL.com](https://www.aol.com/finance/apple-aapl-q1-2026-earnings-233531724.html).

Net: Clearly positive—beats, record iPhone results, strong margin mix, constructive stock reaction.

3) Regulatory issues
- EU DMA/antitrust tensions: Ongoing disputes with the European Commission over App Store rules (anti‑steering) and DMA compliance, including reports of a ~€500M fine last year and continued scrutiny; Apple pushed back, calling EU actions “delay tactics” [Yahoo! News](https://uk.news.yahoo.com/apple-accuses-europe-delay-tactics-124701835.html), [WCCFTech](https://wccftech.com/apple-flays-the-eus-lethargy-as-an-alternate-app-store-collapses/), [iClarified](https://www.iClarified.com/99721/apple-accuses-eu-of-political-delay-tactics-to-justify-app-store-fines).
- Broader enforcement climate: European regulators remain active on the privacy–competition nexus, adding to headline risk [Mondaq.Com](https://www.mondaq.com/unitedstates/antitrust-eu-competition/1737130/conflicting-rules-is-a-privacy-savvy-approach-a-threat-to-competition).

Net: Negative—EU scrutiny remains a notable overhang despite operational progress elsewhere.

Internal holdings check (AAPL)
- We hold Apple in:
  - PF001 (US Large Cap Growth): 15,000 shares; market value $2,778,750; unrealized P&L $641,250.
  - PF003 (Diversified Tech Leaders): 25,000 shares; market value $4,631,250; unrealized P&L $756,250.
- Combined exposure: 40,000 shares; market value ~$7,410,000; unrealized P&L ~$1,397,500.
- No AAPL position found in PF002.
From internal database.

### Query 4: Portfolio Strategy Review

Combines internal strategy memos with current market conditions.

In [11]:
query = """Review our Q1 2025 portfolio strategy:
1. What allocation changes did our internal research recommend?
2. What are our top holdings by market value?
3. Any recent news that might affect our strategy?

Provide a summary of whether we should stay the course or adjust."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
# display_citations(result)  # citations are inline in response

Here’s a concise review based on internal research, our current holdings, and recent market/news flow.

1) Allocation changes recommended (Q1 2025)
- Increase: NVDA +3%, META +2%, PLTR +1
- Maintain: MSFT, AAPL
- Reduce: AMD −1%, CRM −1
According to internal research (strategy memo, Jan 5, 2025). Risk memo (Jan 10, 2025) flagged valuations/regulatory as key risks and suggested QQQ put-spread hedges and maintaining 5–10% cash.

2) Top holdings by market value (firmwide)
- NVDA: $17.51M
- MSFT: $9.56M
- AAPL: $7.41M
- CRM: $3.26M
- META: $2.63M
Next: AVGO $2.14M, ORCL $2.10M, PLTR $1.63M, AMZN $1.35M, AMD $1.16M. From internal database (PF001–PF003).

3) Recent news that could affect the strategy
- NVIDIA: Ongoing China restrictions remain a headline risk, but demand/pricing stay tight; reports of illicit shipments underscore supply scarcity and strong demand signals even under export controls [MSN](https://www.msn.com/en-us/money/other/160m-in-nvidia-ai-chips-secretly-shipped-to-china-in-new-york-scheme/ss-AA1VgTKz?ocid=finance-verthp-feeds), while competitive dynamics and export constraints are cited as potential margin headwinds [Yahoo! Finance](https://finance.yahoo.com/news/look-advanced-micro-devices-amd-091020341.html).
- Microsoft: Strong AI-driven cloud demand with surging capex; near-term investor debate is capex pace vs. long-term AI positioning and supply constraints [Yahoo! Finance](https://finance.yahoo.com/news/msft-q4-deep-dive-ai-053430582.html), [MSN](https://www.msn.com/en-us/money/companies/microsoft-delivers-strong-q2-but-surging-capex-in-line-azure-growth-prompts-pullback-analysts/ar-AA1Vfv1i?ocid=finance-verthp-feeds), [CNBC](https://www.cnbc.com/2026/01/29/analysts-look-past-azure-miss-as-microsoft-slides-post-earnings.html).
- Apple: Better-than-expected results on iPhone and Services; AI partnerships (Siri–Gemini) cited as catalysts; China rebound helps near term [CNBC](https://www.cnbc.com/2026/01/29/apple-earnings-what-top-analysts-expect-to-see-in-the-quarterly-report.html), [Yahoo! News](https://sg.finance.yahoo.com/news/apple-sales-profit-beat-wall-213143171.html), [PYMNTS.com](https://www.pymnts.com/earnings/2026/apple-signals-ai-will-power-payments-security-and-growth/).
- Meta: Strong ad momentum and growing AI commercialization, but Reality Labs losses and rising capex may pressure FCF and margins; regulatory scrutiny persists [Yahoo! Finance](https://finance.yahoo.com/news/meta-stock-jumps-ai-book-161325932.html), [AOL.com](https://www.aol.com/finance/everyone-loves-meta-platforms-again-171921681.html).
- AMD: MI300/MI350 ramp gaining traction with hyperscalers and OCI; bullish data center outlook, but still chasing NVIDIA’s ecosystem; earnings catalyst near term [Nasdaq](https://www.nasdaq.com/articles/amds-ai-everywhere-everyone-push-stock-ready-rally), [Yahoo! Finance](https://finance.yahoo.com/news/intel-post-earnings-selloff-just-192653240.html).
- Salesforce: Agentforce narrative constructive, but revenue growth deceleration and cost/price dynamics cloud margin trajectory; investor skepticism lingered after share weakness [Yahoo! Finance](https://finance.yahoo.com/news/bull-case-salesforce-crm-could-231217083.html), [AOL.com](https://www.aol.com/finance/salesforce-deepens-defense-wildfire-ai-131258579.html).

Recommendation: stay the course with targeted adjustments
- Stay overweight AI leaders per plan: Maintain/incrementally add NVDA, META, PLTR toward targets where underweight, mindful of NVDA export-control risk and META capex/Reg risk.
- Maintain MSFT and AAPL weights; both have durable AI monetization optionality, though MSFT capex implies near-term volatility.
- Trim/keep AMD and CRM at the reduced targets given valuation/execution (AMD) and growth/margin uncertainties (CRM), in line with internal guidance.
- Risk posture: Given elevated valuations and regulatory overhangs, keep a 5–10% cash buffer and consider QQQ put-spread hedges as per the risk memo. According to internal research and the internal database.

---

## 🔟 Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**

| Query Type | Tool Used |
|------------|----------|
| Portfolio positions | `internal_query_database` |
| Internal research | `internal_search_research` |
| Company lookup | `bigdata_lookup_company` |
| Market news | `bigdata_search_news` |

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

---

## 🎯 Next Steps

**Extend this architecture:**

- Add More External Data Sources 
- Implement Distributed Cache for company lookup and Response Caching
   


**Production Considerations:**
- Implement proper error handling for API failures
- Add rate limiting to manage API costs
- Cache frequently requested data
- Monitor performance in LangSmith

- **Advanced Graph Patterns, based on need**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

- **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

- **Context Compression**
   - Strategy to compress the context (i.e. summarizing when context window reaches 80%)

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **Search API Reference**: https://docs.bigdata.com/search-api
- **Knowledge Graph API**: https://docs.bigdata.com/knowledge-graph
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com

## 📊 LangSmith Tracing

![LangSmith Tracing](./static/langsmith_search.png)

---